In [2]:

import os
import argparse
import json
from dataclasses import dataclass
from typing import List, Dict, Any
import torch
from torch.utils.data import Dataset

import pandas as pd
from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import bitsandbytes as bnb  # noqa: F401 (needed for 4-bit/8-bit quantization)

ModuleNotFoundError: No module named 'torch'

In [ ]:
LETTER_VOCAB = ["A", "B", "C", "D", "E"]


def build_prompt(options: List[str], legend: bool) -> str:
    text = (
        "I am showing you five apartment floorplans, labeled A through E.\n"
        "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
        "The thick black outline of each of the floorplans indicates the boundary of that floorplan. "
        "The red bar drawn on the black outline of each of the floorplans marks the main entrance of that floorplan.\n"
    )
    if legend:
        text += "There is a color legend indicating the color-coding of room types below all the floorplans.\n"
    text += (
        "Examine each floorplan only within its thick black outer boundary, focusing on spatial layout, room types, and relative sizes.\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern?\n\n"
        "Answer with a single letter only (A/B/C/D/E)."
    )
    return text


class OddOneOutDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_root: str, processor: AutoProcessor, legend: bool = False):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.processor = processor
        self.legend = legend

    def __len__(self):
        return len(self.df)

    def _messages(self, image: Image.Image, prompt_text: str, answer_text: str):
        # Qwen2.5-VL chat format: list of {role, content=[{type:"text"/"image", ...}, ...]}
        user_part = [
            {"type": "text", "text": prompt_text},
            {"type": "image", "image": image},
        ]
        assistant_part = [{"type": "text", "text": answer_text}]
        return [
            {"role": "user", "content": user_part},
            {"role": "assistant", "content": assistant_part},
        ]

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        # Build path like example_{orig_index}_2.png
        img_name = f"example_{row['orig_index']}_2.png"
        img_path = os.path.join(self.image_root, img_name)
        if not os.path.exists(img_path):
            # Also try absolute path if user placed files differently
            alt_path = os.path.join(self.image_root, "datasets", "easy", img_name)
            if os.path.exists(alt_path):
                img_path = alt_path

        image = Image.open(img_path).convert("RGB")

        prompt_text = build_prompt(LETTER_VOCAB, legend=self.legend)
        answer_text = str(row["outlier_id"]).strip()

        # For efficiency, we pre-create the templated texts and keep the raw PIL image here.
        # Collator will tokenize/pad and create labels with the prompt masked.
        # The 'prompt_only_text' corresponds to user message only, as used for masking.
        messages_full = self._messages(image, prompt_text, answer_text)
        messages_prompt_only = [{"role": "user", "content": messages_full[0]["content"]}]

        # Convert chat templates to plain strings (with special tokens) up-front
        text_full = self.processor.apply_chat_template(messages_full, tokenize=False)
        text_prompt_only = self.processor.apply_chat_template(messages_prompt_only, tokenize=False)

        return {
            "image": image,
            "text_full": text_full,
            "text_prompt": text_prompt_only,
            "label_letter": answer_text,
            "idx": int(row["orig_index"]),
        }


@dataclass
class VLDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Unpack
        images = [f["image"] for f in features]
        text_full_list = [f["text_full"] for f in features]
        text_prompt_list = [f["text_prompt"] for f in features]

        # Process the pair (text + images) together to get input_ids, pixel_values, etc.
        batch_inputs = self.processor(
            text=text_full_list,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        tokenizer = self.processor.tokenizer

        # Build labels where prompt tokens are masked (-100)
        labels_list = []
        for tfull, tprompt in zip(text_full_list, text_prompt_list):
            ids_full = tokenizer(tfull, add_special_tokens=False).input_ids
            ids_prompt = tokenizer(tprompt, add_special_tokens=False).input_ids
            lab = ids_full.copy()
            lab[: len(ids_prompt)] = [-100] * len(ids_prompt)
            labels_list.append(torch.tensor(lab, dtype=torch.long))

        # Pad labels to the same length as input_ids
        # Use the tokenizer pad to keep token type consistency
        labels_padded = tokenizer.pad(
            {"input_ids": labels_list},
            padding="longest",
            return_tensors="pt",
        )["input_ids"]

        batch_inputs["labels"] = labels_padded
        return batch_inputs


def get_model_and_processor(model_name: str, bnb_4bit: bool = True):
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

    if bnb_4bit:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            load_in_4bit=True,
            quantization_config=dict(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            ),
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )

    # Attach LoRA adapters on the language model submodules.
    # NOTE: You may need to tweak target_modules to match your installed
    # transformers/model version. Common linear projection names are used here.
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        # If your model exposes a vision-text projector (e.g., mm_projector),
        # include it in modules_to_save so it remains trainable:
        modules_to_save=None,  # e.g., ["mm_projector"]
    )
    model = get_peft_model(model, lora_cfg)
    return model, processor


def run_eval(model, processor, df_eval: pd.DataFrame, image_root: str, legend: bool, max_samples: int = None):
    model.eval()
    ds = OddOneOutDataset(df_eval if max_samples is None else df_eval.iloc[:max_samples], image_root, processor, legend)
    correct = 0
    total = 0
    preds, gts, idxs = [], [], []

    for i in range(len(ds)):
        ex = ds[i]
        # Build inputs for generation
        inputs = processor(
            text=[ex["text_prompt"]],  # generation prompt (user-only)
            images=[ex["image"]],
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            gen = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                num_beams=1,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        out = processor.tokenizer.decode(gen[0], skip_special_tokens=True)
        # Extract the first valid letter A-E from the generated text
        pred_letter = None
        for ch in out:
            if ch in LETTER_VOCAB:
                pred_letter = ch
                break

        gt = ex["label_letter"]
        if pred_letter == gt:
            correct += 1
        total += 1
        preds.append(pred_letter if pred_letter is not None else "")
        gts.append(gt)
        idxs.append(ex["idx"])

    acc = correct / max(1, total)
    return {"accuracy": acc, "preds": preds, "gts": gts, "idxs": idxs}